# Load the data

Loads in the dataset into our notebook and checks the first 5 rows to make sure it works

In [ ]:
import os
import pandas as pd
from dotenv import load_dotenv
from supabase import create_client

load_dotenv()

url = os.getenv("SUPABASE_URL")
key = os.getenv("SUPABASE_KEY")
supabase = create_client(url, key)

# Allows us to fetch more than the 1 000 row limit that SupaBase has
all_rows = []
step = 1000
start = 0

while True:
    res = (
        supabase.table("telco_churn")
        .select("*")
        .range(start, start + step - 1)
        .execute()
    )
    if not res.data:
        break
    all_rows.extend(res.data)
    start += step

df = pd.DataFrame(all_rows)

df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## EDA 

Exploratory Data Anaylsis, we're trying to grasp why some customers churn and why some choose to stay. Are there any possible correlations or causations?

In [8]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


df['TotalCharges_clean'] = df['TotalCharges'].str.strip().replace('', '0.0').astype(float)

num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges_clean']
num_summary = df[num_cols].describe().T
print("--- Numerical Summary ---")
print(num_summary)

churn_counts = df['Churn'].value_counts(normalize=True) * 100
print("\n--- Churn Rate ---")
print(churn_counts)

contract_churn = df.groupby('Contract')['Churn'].value_counts(normalize=True).unstack() * 100
print("\n--- Churn by Contract (%) ---")
print(contract_churn)

internet_churn = df.groupby('InternetService')['Churn'].value_counts(normalize=True).unstack() * 100
print("\n--- Churn by Internet Service (%) ---")
print(internet_churn)

payment_churn = df.groupby('PaymentMethod')['Churn'].value_counts(normalize=True).unstack() * 100
print("\n--- Churn by Payment Method (%) ---")
print(payment_churn)

--- Numerical Summary ---
                     count         mean          std    min     25%      50%  \
tenure              7043.0    32.371149    24.559481   0.00    9.00    29.00   
MonthlyCharges      7043.0    64.761692    30.090047  18.25   35.50    70.35   
TotalCharges_clean  7043.0  2279.734304  2266.794470   0.00  398.55  1394.55   

                        75%      max  
tenure                55.00    72.00  
MonthlyCharges        89.85   118.75  
TotalCharges_clean  3786.60  8684.80  

--- Churn Rate ---
Churn
No     73.463013
Yes    26.536987
Name: proportion, dtype: float64

--- Churn by Contract (%) ---
Churn                  No        Yes
Contract                            
Month-to-month  57.290323  42.709677
One year        88.730482  11.269518
Two year        97.168142   2.831858

--- Churn by Internet Service (%) ---
Churn                   No        Yes
InternetService                      
DSL              81.040892  18.959108
Fiber optic      58.107235  41.8927